In [3]:
# (Colab comes with requests, beautifulsoup4, and pandas pre-installed, but this ensures everything is ready)
import requests
from bs4 import BeautifulSoup
import pandas as pd

print("Libraries imported successfully!.")

Libraries imported successfully!.


In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Starting URL
url = "https://books.toscrape.com/catalogue/page-1.html"

book_data = []

# Scrape multiple pages
while url:

    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        books = soup.find_all("article", class_="product_pod")

        for book in books:
            title = book.find("h3").find("a")["title"]

            price = book.find(
                "p", class_="price_color"
            ).text

            availability = book.find(
                "p", class_="instock availability"
            ).text.strip()

            book_data.append({
                "Book Title": title,
                "Price": price,
                "Availability": availability
            })

        print(f"Scraped {len(books)} books from {url}")

        # Find the next page
        next_button = soup.find("li", class_="next")

        if next_button:
            next_page = next_button.find("a")["href"]

            from urllib.parse import urljoin
            url = urljoin(url, next_page)

        else:
            url = None

    except requests.exceptions.RequestException as e:
        print("Website request failed:", e)
        break

# Create dataset
df = pd.DataFrame(book_data)

print("\nTotal books scraped:", len(df))

# Display dataset
print(df.head())

# Save dataset
df.to_csv("custom_scraped_books_dataset.csv", index=False)

print("\nDataset successfully saved!")

Scraped 20 books from https://books.toscrape.com/catalogue/page-1.html
Scraped 20 books from https://books.toscrape.com/catalogue/page-2.html
Scraped 20 books from https://books.toscrape.com/catalogue/page-3.html
Scraped 20 books from https://books.toscrape.com/catalogue/page-4.html
Scraped 20 books from https://books.toscrape.com/catalogue/page-5.html
Scraped 20 books from https://books.toscrape.com/catalogue/page-6.html
Scraped 20 books from https://books.toscrape.com/catalogue/page-7.html
Scraped 20 books from https://books.toscrape.com/catalogue/page-8.html
Scraped 20 books from https://books.toscrape.com/catalogue/page-9.html
Scraped 20 books from https://books.toscrape.com/catalogue/page-10.html
Scraped 20 books from https://books.toscrape.com/catalogue/page-11.html
Scraped 20 books from https://books.toscrape.com/catalogue/page-12.html
Scraped 20 books from https://books.toscrape.com/catalogue/page-13.html
Scraped 20 books from https://books.toscrape.com/catalogue/page-14.html
S

In [9]:
import pandas as pd

# 1. Load the saved dataset back into a DataFrame
df = pd.read_csv("custom_scraped_books_dataset.csv")

# 2. Clean the 'Price' column using regex to remove any non-numeric/currency characters
df["Cleaned Price"] = (
    df["Price"].str.replace(r"[^\d.]", "", regex=True).astype(float)
)

# 3. View basic statistics of your scraped dataset
print("--- Dataset Summary Statistics ---")
print(df.describe())

# 4. Display the updated dataset with the clean price column
print("\n--- Cleaned Data Preview ---")
print(df[["Book Title", "Cleaned Price", "Availability"]].head())

# 5. Save the final cleaned dataset
df.to_csv("cleaned_books_dataset.csv", index=False)
print("\nCleaned dataset saved successfully as 'cleaned_books_dataset.csv'!")

--- Dataset Summary Statistics ---
       Cleaned Price
count     1000.00000
mean        35.07035
std         14.44669
min         10.00000
25%         22.10750
50%         35.98000
75%         47.45750
max         59.99000

--- Cleaned Data Preview ---
                              Book Title  Cleaned Price Availability
0                   A Light in the Attic          51.77     In stock
1                     Tipping the Velvet          53.74     In stock
2                             Soumission          50.10     In stock
3                          Sharp Objects          47.82     In stock
4  Sapiens: A Brief History of Humankind          54.23     In stock

Cleaned dataset saved successfully as 'cleaned_books_dataset.csv'!
